# 04e · Six-method proposals → human-review ZIP

Combines the five methods in 04c with the portable **facebook/sam3** adapter
from 04d. No training, automatic winner selection, forced object count or
robot execution. The output is a **proposal bank, not ground truth**.

Start a **fresh Colab runtime**, preferably GPU. CPU is supported but slow.
One model pipeline is loaded at a time; RAM/VRAM needs still apply. SAM 3
uses CPU float32 / CUDA float16, not SAM 3.1 or an A100-only implementation.
Add your approved Hugging Face read token as the Colab secret **HF_TOKEN**.

## Input: your complete `/data` tree, including new photos and subfolders

On your PC run `python demos/roboqc/colab/prepare_04e_inputs.py` from the
project root. This packages existing images only (no model execution) as
`demos/roboqc/colab/assets/data_04e_inputs.zip`. Upload that ZIP in the input
cell, or put it in Drive and set `INPUT_PATH` to its full Drive path.
An existing Drive image folder also works. Recursive discovery includes
uppercase extensions. Relative paths and originals are preserved; identical
filenames in different folders cannot overwrite one another.

Empty `INPUT_PATH` opens an upload, saves a new copy of the input ZIP in
Drive, then reads that Drive copy. On later sessions use the printed path.
Input ZIPs are copied/extracted locally for speed; weights stay local too.
Outputs/checkpoints and the final ZIP are stored in Drive.

All images run by default. With hundreds of images and multiple prompts,
this can take hours and produce a large archive. `MAX_IMAGES=3` is a useful
smoke test; set it back to `None` for the full batch. No image bytes are
embedded in this notebook. Check data/model redistribution rights.


In [ ]:
# Change these values BEFORE running subsequent cells.
USE_DRIVE = True
ROOT = '/content/drive/MyDrive/Robotekin2026'
SEED = 42
DATASET_PATH = ''  # Existing dataset folder on Drive; empty opens upload.
MODEL_PATH = ''    # Existing exported model folder; empty opens upload.
RESUME_RUN = ''    # Existing run folder for checkpoint continuation.
INPUT_PATH = ''  # Drive ZIP or image folder; blank uploads and saves it in Drive
METHODS = ['yoloe_text','sam2_auto','grounding_dino_sam2','owlv2_sam2','florence2_sam2','sam3_text']
PROMPTS = ['metal bracket','sheet metal part','metal mounting clip']
ROI_NORM = None  # Optional common [left,top,right,bottom] fractions
MAX_IMAGES = None  # None = ALL recursively discovered images
DEVICE_CHOICE = 'auto'  # auto, cpu, cuda
MODEL_CACHE = '/content/roboqc_review_cache'  # Never Drive
SAM2_ID = 'facebook/sam2.1-hiera-large'  # small variant can reduce memory
SAM2_COMMIT = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
MODEL_REVISIONS = {}  # HF model ID -> pinned revision; resolved revisions exported
YOLOE_CHECKPOINT = 'yoloe-26l-seg.pt'
IMGSZ = 960
SAM3_THRESHOLD = 0.40
MIN_MASK_AREA = 100
MAX_MASK_FRACTION = 0.85
RESUME_BUNDLE = ''  # New run; set an existing bundle folder to resume
PREVIEW_IMAGES = 3
MAX_EXTRACT_GB = 10
MAX_EXTRACT_FILES = 50000


In [ ]:
import os, random, subprocess, sys, uuid
from datetime import datetime, timezone
from pathlib import Path
print('Python:', sys.version, '| executable:', sys.executable, flush=True)
install_log = Path('/content/roboqc_pip_install.log')
install_log.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'numpy>=1.26,<3', 'torch>=2.5.1,<3', 'torchvision>=0.20.1,<1', 'transformers==5.2.0', 'huggingface-hub>=1.3,<2', 'safetensors>=0.5,<1', 'ultralytics>=8.4.0,<9', 'matplotlib>=3.9,<4', 'Pillow>=10,<13', 'opencv-python-headless>=4.10,<6', 'pandas>=2.2,<4', 'scipy>=1.13,<2', 'hydra-core>=1.3.2,<2', 'iopath>=0.1.10,<1', 'tqdm>=4.66,<5']
with install_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'pip failed (exit {exit_code}). Read the ERROR lines above. Full log: {install_log}')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    ROOT = '/content/Robotekin2026'
RUN = Path(RESUME_RUN) if RESUME_RUN else Path(ROOT) / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True, exist_ok=bool(RESUME_RUN))
import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
print('Device:', DEVICE, '| Run:', RUN)
(RUN / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


In [ ]:
import gc, hashlib, json, shutil, time, traceback
import pandas as pd
from PIL import Image, ImageOps
if not USE_DRIVE: raise ValueError('04e requires Drive for persistent inputs and exports.')
if DEVICE_CHOICE not in ['auto','cpu','cuda']: raise ValueError('Invalid DEVICE_CHOICE')
if DEVICE_CHOICE == 'cuda' and not GPU: raise ValueError('CUDA is unavailable')
if DEVICE_CHOICE != 'auto': DEVICE = DEVICE_CHOICE
GPU = DEVICE == 'cuda'
CACHE = Path(MODEL_CACHE).resolve()
if CACHE.is_relative_to(Path('/content/drive').resolve()): raise ValueError('Weights must stay on local disk')
CACHE.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME'] = str(CACHE / 'huggingface')
os.environ['SAM2_BUILD_CUDA'] = '0'
if any(m == 'sam2_auto' or m.endswith('_sam2') for m in METHODS):
    command = [sys.executable,'-m','pip','install','--no-build-isolation',
               'git+https://github.com/facebookresearch/sam2.git@' + SAM2_COMMIT]
    with install_log.open('a',encoding='utf-8') as log:
        process = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line)
        if process.wait(): raise RuntimeError(f'SAM 2 installation failed; see {install_log}')
if 'sam3_text' in METHODS:
    from huggingface_hub import login
    from getpass import getpass
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception: pass
    if not token: token = getpass('Approved facebook/sam3 read token (hidden): ')
    if not token: raise ValueError('SAM 3 requires an approved account/token')
    login(token=token,add_to_git_credential=False)
    del token
print('Device:',DEVICE,'| SAM 3 dtype:', 'float16' if GPU else 'float32')
if not GPU: print('WARNING: CPU batch inference may be very slow.')


In [ ]:
SOURCES = {'roboqc/__init__.py': '', 'roboqc/industrial/__init__.py': '', 'roboqc/industrial/vision_inference.py': '"""CPU segmentation, planar geometry and optional visible-quality classifier.\n\nThis module intentionally imports no GUI or robot library and is embedded in Colab.\n"""\n\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nCLASSES = ["bolt", "nut", "washer", "bracket"]\nQUALITY = ["good", "defect_1", "defect_2"]\n\n\ndef mask_geometry(mask, symmetric=False, clearance_px=3):\n    mask = np.asarray(mask, dtype=np.uint8)\n    ys, xs = np.nonzero(mask)\n    if len(xs) < 3:\n        return {"center_px": None, "orientation_deg": None, "grasp_candidates": []}\n    center = [float(xs.mean()), float(ys.mean())]\n    eigenvalues, eigenvectors = np.linalg.eigh(np.cov(np.column_stack([xs, ys]), rowvar=False))\n    orientation = None\n    if not symmetric and eigenvalues[-1] > 1.3 * max(eigenvalues[0], 1e-9):\n        axis = eigenvectors[:, -1]\n        orientation = float(np.degrees(np.arctan2(axis[1], axis[0])) % 180)\n    distance = cv2.distanceTransform(mask, cv2.DIST_L2, 5)\n    _, radius, _, point = cv2.minMaxLoc(distance)\n    candidates = []\n    if radius >= clearance_px:\n        candidates.append({"position_px": list(point), "clearance_px": float(radius),\n                           "orientation_deg": orientation, "validated": False})\n    return {"center_px": center, "orientation_deg": orientation, "grasp_candidates": candidates}\n\n\ndef classify_mask_color(frame, mask, minimum_fraction=.6):\n    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)\n    # Erode edges to reduce background contamination; classify only saturated pixels.\n    interior = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0\n    values = hsv[interior]\n    if not len(values):\n        return None, 0.0\n    ranges = {"red": ((values[:, 0] <= 8) | (values[:, 0] >= 170)),\n              "green": (values[:, 0] >= 35) & (values[:, 0] <= 85),\n              "blue": (values[:, 0] >= 86) & (values[:, 0] <= 130)}\n    votes = {key: float(np.mean(region & (values[:, 1] >= 70) & (values[:, 2] >= 45))) for key, region in ranges.items()}\n    color = max(votes, key=votes.get)\n    confidence = votes[color]\n    return (color if confidence >= minimum_fraction else None), confidence\n\n\ndef quality_crop(frame, mask):\n    ys, xs = np.nonzero(mask)\n    if not len(xs):\n        raise ValueError("Empty instance mask")\n    patch = np.where(mask[..., None] > 0, frame, 127)[ys.min():ys.max()+1, xs.min():xs.max()+1]\n    return cv2.cvtColor(cv2.resize(patch, (224, 224)), cv2.COLOR_BGR2RGB)\n\n\nclass Segmenter:\n    def __init__(self, bundle, prefer_onnx=True, confidence=.35):\n        from ultralytics import YOLO\n        self.root = Path(bundle)\n        self.metadata = json.loads((self.root / "vision_manifest.json").read_text(encoding="utf-8"))\n        if self.metadata.get("schema_version") != "industrial-vision-1.0":\n            raise ValueError("Unsupported vision artifact schema")\n        self.classes = self.metadata["classes"]\n        filename = "segmentation.onnx" if prefer_onnx and (self.root / "segmentation.onnx").exists() else "segmentation.pt"\n        if filename.endswith(".pt") and self.metadata.get("model_family") == "yoloe":\n            from ultralytics import YOLOE\n            self.model = YOLOE(str(self.root / filename), task="segment")\n        else:\n            self.model = YOLO(str(self.root / filename), task="segment")\n        self.confidence = confidence\n        self.quality_session = None\n        if self.metadata.get("quality_enabled"):\n            import onnxruntime as ort\n            self.quality_session = ort.InferenceSession(str(self.root / "quality.onnx"), providers=["CPUExecutionProvider"])\n\n    def predict(self, frame):\n        # PT otherwise uses minimal rectangular padding, unlike static square ONNX.\n        # YOLOE also defaults to agnostic NMS while YOLO(onnx) does not.\n        prediction = self.model.predict(frame, device="cpu", imgsz=self.metadata.get("imgsz", 640),\n            conf=self.confidence, rect=False, agnostic_nms=self.metadata.get("model_family") == "yoloe",\n            iou=.7, max_det=300, retina_masks=False, augment=False, verbose=False)[0]\n        objects = []\n        if prediction.masks is None:\n            return objects\n        masks = prediction.masks.data.cpu().numpy()\n        height, width = frame.shape[:2]\n        for i, (box, cls, confidence) in enumerate(zip(prediction.boxes.xyxy.cpu().numpy(), prediction.boxes.cls.cpu().numpy(), prediction.boxes.conf.cpu().numpy())):\n            # Ultralytics masks may be in letterboxed coordinates, not frame coordinates.\n            from ultralytics.utils.ops import scale_masks\n            import torch\n            restored = scale_masks(torch.as_tensor(masks[i:i+1, None],dtype=torch.float32), (height, width))[0, 0].numpy()\n            mask = (restored > .5).astype(np.uint8)\n            if not mask.any():\n                continue\n            x1, y1, x2, y2 = box.tolist()\n            if int(cls) != cls or not 0 <= int(cls) < len(self.classes):\n                raise ValueError("Prediction class ID does not match the exported class mapping")\n            object_type = self.classes[int(cls)]\n            color, color_confidence = classify_mask_color(frame, mask)\n            geometry = mask_geometry(mask, symmetric=object_type in ("washer", "nut"))\n            if geometry["center_px"] is None:\n                continue\n            quality, quality_confidence = "unknown", 0.0\n            if self.quality_session is not None:\n                crop = quality_crop(frame, mask).astype(np.float32) / 255\n                x = ((crop - [.485, .456, .406]) / [.229, .224, .225]).transpose(2, 0, 1)[None].astype(np.float32)\n                logits = self.quality_session.run(None, {self.quality_session.get_inputs()[0].name: x})[0][0]\n                p = np.exp(logits - logits.max()); p /= p.sum()\n                quality_confidence = float(p.max())\n                if quality_confidence >= self.metadata.get("quality_threshold", .8):\n                    quality = QUALITY[int(p.argmax())]\n            partial = x1 <= 1 or y1 <= 1 or x2 >= width-1 or y2 >= height-1\n            if partial:\n                quality = "unknown"\n                geometry["grasp_candidates"] = []\n            objects.append({"type": object_type, "confidence": float(confidence), "mask": mask,\n                "bbox": [int(x1), int(y1), int(x2-x1), int(y2-y1)], "color": color,\n                "color_confidence": color_confidence, "quality": quality,\n                "quality_confidence": quality_confidence, "partial": partial, **geometry})\n        return objects\n\n\ndef draw_instances(frame, objects):\n    out = frame.copy()\n    for i, obj in enumerate(objects):\n        mask = obj["mask"] > 0\n        color = np.array([(53*i+60) % 255, (97*i+100) % 255, (131*i+180) % 255])\n        out[mask] = (.55*out[mask] + .45*color).astype(np.uint8)\n        x, y, w, h = obj["bbox"]\n        cv2.rectangle(out, (x, y), (x+w, y+h), color.tolist(), 2)\n        cv2.putText(out, f"{obj[\'type\']} {obj[\'confidence\']:.2f} {obj[\'quality\']}", (x, max(16,y-5)), cv2.FONT_HERSHEY_SIMPLEX, .45, (255,255,255), 1)\n        for point in obj["grasp_candidates"]:\n            cv2.circle(out, tuple(map(int, point["position_px"])), 4, (0,255,255), -1)\n    return out\n', 'roboqc/industrial/sam_masks.py': '"""SAM candidate export utilities. No model loading, downloads or inference."""\n\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom .vision_inference import mask_geometry\n\n\ndef save_candidates(rgb, annotations, output, minimum_area=100, maximum_fraction=.85, seed=42):\n    """Preserve all raw masks; export filtered candidates without claiming object labels."""\n    rgb = np.asarray(rgb)\n    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:\n        raise ValueError("Expected a uint8 RGB image")\n    if minimum_area < 1 or not 0 < maximum_fraction <= 1:\n        raise ValueError("Require minimum_area >= 1 and 0 < maximum_fraction <= 1")\n    height,width = rgb.shape[:2]\n    masks = [np.asarray(a["segmentation"],dtype=bool) for a in annotations]\n    if any(m.shape != (height,width) for m in masks):\n        raise ValueError("Every mask must match the original image shape")\n    out = Path(output)\n    out.mkdir(parents=True,exist_ok=False)\n    np.savez_compressed(out / "raw_masks.npz",masks=np.stack(masks) if masks else np.zeros((0,height,width),bool))\n    rows = []\n    for index,(annotation,mask) in enumerate(zip(annotations,masks)):\n        area = int(mask.sum())\n        reason = "below_minimum_area" if area < minimum_area else "above_maximum_fraction" if area > maximum_fraction*height*width else None\n        ys,xs = np.nonzero(mask)\n        box = [int(xs.min()),int(ys.min()),int(xs.max()-xs.min()+1),int(ys.max()-ys.min()+1)] if area else None\n        filename = f"mask_{index:04d}.png" if reason is None else None\n        if filename:\n            if not cv2.imwrite(str(out / filename),mask.astype(np.uint8)*255):\n                raise OSError(f"Could not save mask: {out / filename}")\n        rows.append({"id":index,"raw_mask_index":index,"area_px":area,"bbox_xywh":box,\n            "model_predicted_iou":float(annotation["predicted_iou"]) if annotation.get("predicted_iou") is not None else None,\n            "stability_score":float(annotation["stability_score"]) if annotation.get("stability_score") is not None else None,\n            "detector_score":float(annotation["detector_score"]) if annotation.get("detector_score") is not None else None,\n            "prompt_label":annotation.get("prompt_label"),\n            "kept":reason is None,"filter_reason":reason,"mask_file":filename,\n            "category":"unknown","quality":"unknown",**mask_geometry(mask)})\n    colors = np.random.default_rng(seed).integers(40,240,size=(len(rows),3))\n    overlay = rgb.copy()\n    # Smaller nested regions are drawn last, but all original overlapping masks remain separate.\n    for row in sorted((r for r in rows if r["kept"]),key=lambda r:r["area_px"],reverse=True):\n        mask = masks[row["id"]]\n        overlay[mask] = (.55*overlay[mask] + .45*colors[row["id"]]).astype(np.uint8)\n    for row in rows:\n        if row["kept"]:\n            x,y,w,h = row["bbox_xywh"]\n            cv2.rectangle(overlay,(x,y),(x+w-1,y+h-1),(0,170,0),1)\n            cv2.putText(overlay,str(row["id"]),(x,max(14,y)),cv2.FONT_HERSHEY_SIMPLEX,.5,(180,0,0),1)\n    if not cv2.imwrite(str(out / "overlay.png"),cv2.cvtColor(overlay,cv2.COLOR_RGB2BGR)):\n        raise OSError("Could not save overlay")\n    result = {"schema_version":"sam-candidates-1.0","image_size":[height,width],\n        "raw_masks":len(rows),"kept_candidates":sum(r["kept"] for r in rows),"candidates":rows,\n        "filters":{"minimum_area":minimum_area,"maximum_fraction":maximum_fraction},\n        "accuracy":"not_measured","warning":"Candidates may overlap or describe subparts/background, not unique physical objects"}\n    (out / "candidates.json").write_text(json.dumps(result,indent=2),encoding="utf-8")\n    return result,overlay\n', 'roboqc/industrial/pretrained_compare.py': '"""Manual Colab comparison adapters. No imports load models or start inference."""\n\nimport hashlib\nimport os\nfrom pathlib import Path\nimport tempfile\nimport warnings\n\nimport numpy as np\n\nDEFAULT_METHODS = ["yoloe_text","sam2_auto","grounding_dino_sam2","owlv2_sam2","florence2_sam2"]\nMODEL_IDS = {"grounding_dino_sam2":"IDEA-Research/grounding-dino-base",\n             "owlv2_sam2":"google/owlv2-base-patch16-ensemble",\n             "florence2_sam2":"florence-community/Florence-2-base",\n             "sam3_text":"facebook/sam3"}\n\n\ndef crop_image(rgb, normalized_roi=None):\n    """A shared optional workspace crop; no method gets a different region."""\n    height,width = rgb.shape[:2]\n    if normalized_roi is None:\n        return rgb,(0,0,width,height)\n    roi = np.asarray(normalized_roi,dtype=float)\n    if roi.shape != (4,) or not np.isfinite(roi).all() or not (0 <= roi[0] < roi[2] <= 1 and 0 <= roi[1] < roi[3] <= 1):\n        raise ValueError("ROI must be normalized [left,top,right,bottom] inside [0,1]")\n    x1,y1,x2,y2 = int(roi[0]*width),int(roi[1]*height),int(roi[2]*width),int(roi[3]*height)\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("ROI is empty at this image resolution")\n    return rgb[y1:y2,x1:x2],(x1,y1,x2,y2)\n\n\ndef restore_masks(annotations, image_shape, roi):\n    x1,y1,x2,y2 = roi\n    restored = []\n    for annotation in annotations:\n        mask = np.asarray(annotation["segmentation"],dtype=bool)\n        if mask.shape != (y2-y1,x2-x1):\n            raise ValueError("Mask dimensions do not match the processed ROI")\n        full = np.zeros(image_shape[:2],bool)\n        full[y1:y2,x1:x2] = mask\n        restored.append(dict(annotation,segmentation=full))\n    return restored\n\n\ndef select_boxes(boxes, scores, labels, size, iou_threshold=.7, maximum=50):\n    """Class-agnostic NMS on detector boxes, not a forced object count."""\n    boxes = np.asarray(boxes,dtype=float).reshape(-1,4)\n    if len(scores) != len(boxes) or len(labels) != len(boxes):\n        raise ValueError("Mismatched detector boxes, scores and labels")\n    width,height = size\n    finite = np.isfinite(boxes).all(axis=1)\n    boxes[:,[0,2]] = boxes[:,[0,2]].clip(0,width)\n    boxes[:,[1,3]] = boxes[:,[1,3]].clip(0,height)\n    valid = [i for i,b in enumerate(boxes) if finite[i] and b[2] > b[0] and b[3] > b[1]\n             and (scores[i] is None or np.isfinite(scores[i]))]\n    order = sorted(valid,key=lambda i:float(scores[i]) if scores[i] is not None else 0,reverse=True)\n    keep = []\n    while order and len(keep) < maximum:\n        i = order.pop(0); keep.append(i)\n        survivors = []\n        for j in order:\n            intersection = np.maximum(0,np.minimum(boxes[i,2:],boxes[j,2:])-np.maximum(boxes[i,:2],boxes[j,:2])).prod()\n            area_i = (boxes[i,2:]-boxes[i,:2]).prod(); area_j = (boxes[j,2:]-boxes[j,:2]).prod()\n            if intersection/max(1,area_i+area_j-intersection) <= iou_threshold:\n                survivors.append(j)\n        order = survivors\n    return boxes[keep], [scores[i] for i in keep], [labels[i] for i in keep]\n\n\ndef mask_metrics(predicted, reference, threshold=.5):\n    """One-to-one mask matching, maximizing qualifying matches before IoU."""\n    from scipy.optimize import linear_sum_assignment\n    if any(np.asarray(m).ndim != 2 for m in [*predicted,*reference]):\n        raise ValueError("Ground truth and predictions must be 2D binary masks")\n    shapes = {np.asarray(m).shape for m in [*predicted,*reference]}\n    if len(shapes) > 1: raise ValueError("Ground-truth masks must match original image dimensions")\n    overlaps = np.zeros((len(reference),len(predicted)))\n    for i,a in enumerate(reference):\n        for j,b in enumerate(predicted):\n            overlaps[i,j] = np.logical_and(a,b).sum()/max(1,np.logical_or(a,b).sum())\n    matched = []\n    if overlaps.size:\n        # A plain maximum-IoU assignment can undercount threshold-qualified matches.\n        bonus = min(overlaps.shape)+1\n        rows,columns = linear_sum_assignment(-((overlaps >= threshold)*bonus+overlaps))\n        matched = [float(overlaps[i,j]) for i,j in zip(rows,columns) if overlaps[i,j] >= threshold]\n    tp = len(matched); fp = len(predicted)-tp; fn = len(reference)-tp\n    return {"mask_iou_threshold":threshold,"tp":tp,"fp":fp,"fn":fn,\n        "precision":tp/(tp+fp) if tp+fp else None,"recall":tp/(tp+fn) if tp+fn else None,\n        "f1":2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else None,\n        "matched_mean_iou":float(np.mean(matched)) if matched else None}\n\n\ndef _sam_model(config, device):\n    import torch\n    from huggingface_hub import hf_hub_download\n    from sam2.build_sam import build_sam2, HF_MODEL_ID_TO_FILENAMES\n    name,filename = HF_MODEL_ID_TO_FILENAMES[config["sam2_id"]]\n    # Large checkpoint reads stay on Colab\'s local disk, not the Drive mount.\n    cache = Path(config.get("sam2_cache",Path(tempfile.gettempdir()) / "roboqc_sam2_cache"))\n    cache.mkdir(parents=True,exist_ok=True)\n    revision = config["revisions"].get(config["sam2_id"],"main")\n    recovered = False\n    for attempt in range(2):\n        path = Path(hf_hub_download(config["sam2_id"],filename,revision=revision,cache_dir=str(cache)))\n        try:\n            state = torch.load(str(path),map_location="cpu",weights_only=True)\n            if not isinstance(state,dict) or not isinstance(state.get("model"),dict) or not state["model"]:\n                raise ValueError("SAM checkpoint must contain a nonempty model state dictionary")\n        except Exception as exc:\n            if attempt:\n                raise RuntimeError(f"SAM checkpoint cannot be read after a fresh download: {path}. "\n                                   "Existing files were preserved; inspect the chained error.") from exc\n            # Never overwrite/delete a possibly damaged cache entry or relax weights_only.\n            cache = Path(tempfile.mkdtemp(prefix="recovery_",dir=cache))\n            warnings.warn(f"SAM checkpoint read failed ({type(exc).__name__}: {exc}); "\n                          f"retrying once in a fresh local cache: {cache}",RuntimeWarning)\n            recovered = True\n            continue\n        break\n    config["sam2_cache"] = str(cache)  # Reuse a recovered download for later methods.\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda:handle.read(1024*1024),b""): digest.update(chunk)\n    # Load the already validated weights once; do not reopen the checkpoint in SAM.\n    model = build_sam2(name,ckpt_path=None,device="cpu",apply_postprocessing=False)\n    model.load_state_dict(state["model"],strict=True)\n    del state\n    return model.to(device).eval(),{"model_id":config["sam2_id"],"revision":path.parent.name,\n        "checkpoint_sha256":digest.hexdigest(),"checkpoint_path":str(path),"cache_recovered":recovered}\n\n\nclass PretrainedMethod:\n    """Load only one method per instance; caller releases it before the next method."""\n    def __init__(self, name, config, device):\n        self.name,self.config,self.device = name,config,device\n        self.provenance = {"method":name,"models":[]}\n        if name == "sam2_auto" or name.endswith("_sam2"):\n            self.sam,source = _sam_model(config,device)\n            self.provenance["models"].append(source)\n            if name == "sam2_auto":\n                from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator\n                self.generator = SAM2AutomaticMaskGenerator(self.sam,points_per_side=config["points_per_side"],\n                    points_per_batch=config["points_per_batch"],crop_n_layers=config["crop_layers"],\n                    pred_iou_thresh=config["sam_iou"],stability_score_thresh=config["sam_stability"],\n                    min_mask_region_area=0,use_m2m=True,output_mode="binary_mask")\n            else:\n                from sam2.sam2_image_predictor import SAM2ImagePredictor\n                self.segmenter = SAM2ImagePredictor(self.sam)\n        if name == "yoloe_text":\n            from ultralytics import YOLOE\n            directory = Path(config["cache"]) / "yoloe"; directory.mkdir(parents=True,exist_ok=True)\n            previous = Path.cwd()\n            try:\n                os.chdir(directory)\n                self.model = YOLOE(config["yoloe_checkpoint"],task="segment")\n                self.model.set_classes(config["prompts"],self.model.get_text_pe(config["prompts"]))\n            finally:\n                os.chdir(previous)\n            checkpoint = directory / Path(config["yoloe_checkpoint"]).name\n            self.provenance["models"].append({"checkpoint":config["yoloe_checkpoint"],\n                "sha256":hashlib.sha256(checkpoint.read_bytes()).hexdigest() if checkpoint.is_file() else None})\n        elif name in MODEL_IDS:\n            from safetensors import SafetensorError\n            from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Florence2ForConditionalGeneration, Sam3Model\n            model_id = MODEL_IDS[name]\n            kwargs = {"cache_dir":config["cache"],"revision":config["revisions"].get(model_id,"main"),"trust_remote_code":False}\n            model_class = Florence2ForConditionalGeneration if name == "florence2_sam2" else Sam3Model if name == "sam3_text" else AutoModelForZeroShotObjectDetection\n            for attempt in range(2):\n                try:\n                    self.processor = AutoProcessor.from_pretrained(model_id,**kwargs)\n                    self.model = model_class.from_pretrained(model_id,use_safetensors=True,**kwargs)\n                except SafetensorError as exc:\n                    if attempt:\n                        raise RuntimeError(f"{model_id} weights remain unreadable after a fresh local cache "\n                                           f"download: {kwargs[\'cache_dir\']}. Previous files were preserved.") from exc\n                    cache = Path(config["cache"]); cache.mkdir(parents=True,exist_ok=True)\n                    kwargs["cache_dir"] = tempfile.mkdtemp(prefix=name + "_recovery_",dir=cache)\n                    warnings.warn(f"{model_id}: {exc}; retrying once in a fresh local cache: "\n                                  f"{kwargs[\'cache_dir\']}",RuntimeWarning)\n                    continue\n                break\n            self.model = self.model.to(device).eval()\n            self.provenance["models"].append({"model_id":model_id,"revision":getattr(self.model.config,"_commit_hash",None),\n                "cache_dir":str(kwargs["cache_dir"]),"cache_recovered":bool(attempt)})\n        elif name != "sam2_auto":\n            raise ValueError(f"Unknown comparison method: {name}")\n\n    def predict(self, rgb):\n        import torch\n        from PIL import Image\n        cfg = self.config; image = Image.fromarray(rgb); height,width = rgb.shape[:2]\n        with torch.inference_mode():\n            if self.name == "sam2_auto":\n                return self.generator.generate(rgb),{"score_type":"SAM predicted IoU; not observed accuracy"}\n            if self.name == "yoloe_text":\n                from ultralytics.utils.ops import scale_masks\n                result = self.model.predict(rgb[:,:,::-1].copy(),device=self.device,imgsz=cfg["imgsz"],\n                    conf=cfg["yoloe_threshold"],rect=False,agnostic_nms=True,iou=cfg["box_nms"],\n                    max_det=cfg["max_detections"],retina_masks=False,verbose=False)[0]\n                if result.masks is None: return [],{"score_type":"YOLO detection confidence"}\n                masks = scale_masks(result.masks.data[:,None].float().cpu(),(height,width))[:,0].numpy() > .5\n                scores = result.boxes.conf.cpu().tolist(); classes = result.boxes.cls.cpu().tolist()\n                annotations = [{"segmentation":m,"detector_score":float(s),"prompt_label":cfg["prompts"][int(c)]}\n                               for m,s,c in zip(masks,scores,classes)]\n                return annotations,{"score_type":"YOLO detection confidence; no predicted mask IoU"}\n            if self.name == "sam3_text":\n                # One concept prompt; use the first configured phrase for this adapter.\n                inputs = self.processor(images=image,text=cfg["prompts"][0],return_tensors="pt").to(self.device)\n                outputs = self.model(**inputs)\n                result = self.processor.post_process_instance_segmentation(outputs,threshold=cfg["sam3_threshold"],\n                    mask_threshold=.5,target_sizes=inputs["original_sizes"].tolist())[0]\n                return [{"segmentation":m.cpu().numpy().astype(bool),"detector_score":float(s),"prompt_label":cfg["prompts"][0]}\n                        for m,s in zip(result["masks"],result["scores"])],{"score_type":"SAM 3 instance score"}\n            boxes,scores,labels = [],[],[]\n            if self.name == "grounding_dino_sam2":\n                inputs = self.processor(images=image,text=". ".join(cfg["prompts"])+".",return_tensors="pt").to(self.device)\n                outputs = self.model(**inputs)\n                result = self.processor.post_process_grounded_object_detection(outputs,inputs["input_ids"],\n                    threshold=cfg["dino_threshold"],text_threshold=cfg["dino_text_threshold"],target_sizes=[(height,width)])[0]\n                boxes,scores,labels = result["boxes"].cpu().numpy(),result["scores"].cpu().tolist(),result["text_labels"]\n            elif self.name == "owlv2_sam2":\n                inputs = self.processor(images=image,text=[cfg["prompts"]],return_tensors="pt").to(self.device)\n                outputs = self.model(**inputs)\n                result = self.processor.image_processor.post_process_object_detection(outputs,threshold=cfg["owl_threshold"],target_sizes=[(height,width)])[0]\n                boxes,scores = result["boxes"].cpu().numpy(),result["scores"].cpu().tolist()\n                labels = [cfg["prompts"][int(i)] for i in result["labels"].cpu().tolist()]\n            elif self.name == "florence2_sam2":\n                for phrase in cfg["prompts"]:\n                    task = "<OPEN_VOCABULARY_DETECTION>"\n                    # BatchFeature casts floating image tensors, keeping token IDs integer.\n                    inputs = self.processor(images=image,text=task+phrase,return_tensors="pt").to(self.device,self.model.dtype)\n                    tokens = self.model.generate(**inputs,max_new_tokens=cfg["florence_max_tokens"],num_beams=3,do_sample=False)\n                    text = self.processor.batch_decode(tokens,skip_special_tokens=False)[0]\n                    result = self.processor.post_process_generation(text,task=task,image_size=image.size)[task]\n                    found = result.get("bboxes",[])\n                    boxes.extend(found); scores.extend([None]*len(found)); labels.extend([phrase]*len(found))\n            before_nms = len(boxes)\n            boxes,scores,labels = select_boxes(boxes,scores,labels,image.size,cfg["box_nms"],cfg["max_detections"])\n            if not len(boxes): return [],{"detector_boxes_before_nms":before_nms,"prompted_boxes":0}\n            self.segmenter.set_image(rgb)\n            annotations = []\n            # Single-box calls avoid ambiguous multi-box/multimask output dimensions.\n            for box,score,label in zip(boxes,scores,labels):\n                masks,qualities,_ = self.segmenter.predict(box=box.astype(np.float32),multimask_output=False)\n                annotations.append({"segmentation":masks[0].astype(bool),"predicted_iou":float(qualities[0]),\n                                    "detector_score":score,"prompt_label":str(label)})\n            return annotations,{"detector_boxes_before_nms":before_nms,"prompted_boxes":len(boxes),\n                                "score_type":"Detector score and SAM predicted IoU are distinct; Florence detector scores unavailable"}\n', 'roboqc/industrial/sam3.py': '"""SAM 3 image segmentation through Transformers, on CPU or CUDA. No import-time inference."""\n\nfrom pathlib import Path\nimport tempfile\nfrom types import SimpleNamespace\nimport warnings\n\nimport numpy as np\n\nMODEL_ID = "facebook/sam3"\n\n\ndef select_runtime(device="auto"):\n    import torch\n    if device not in ["auto","cpu","cuda"]: raise ValueError("DEVICE_CHOICE must be auto, cpu or cuda")\n    selected = "cuda" if device == "auto" and torch.cuda.is_available() else "cpu" if device == "auto" else device\n    if selected == "cuda" and not torch.cuda.is_available(): raise ValueError("CUDA requested but unavailable; use auto or cpu")\n    return selected,torch.float16 if selected == "cuda" else torch.float32\n\n\ndef load_predictor(cache, revision="main", device="auto"):\n    from safetensors import SafetensorError\n    from transformers import Sam3Model, Sam3Processor\n    device,dtype = select_runtime(device)\n    cache = Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    kwargs = {"cache_dir":str(cache),"revision":revision,"trust_remote_code":False}\n    for attempt in range(2):\n        try:\n            processor = Sam3Processor.from_pretrained(MODEL_ID,**kwargs)\n            model = Sam3Model.from_pretrained(MODEL_ID,use_safetensors=True,dtype=dtype,\n                attn_implementation="sdpa",**kwargs)\n        except SafetensorError as exc:\n            if attempt:\n                raise RuntimeError(f"SAM 3 checkpoint unreadable after fresh-cache retry: {kwargs[\'cache_dir\']}") from exc\n            kwargs["cache_dir"] = tempfile.mkdtemp(prefix="recovery_",dir=cache)\n            warnings.warn(f"SAM 3 checkpoint read failed: {exc}; retrying in fresh local cache {kwargs[\'cache_dir\']}",RuntimeWarning)\n            continue\n        break\n    model = model.to(device).eval()\n    predictor = SimpleNamespace(model=model,processor=processor,device=device,dtype=dtype)\n    return predictor,{"model_id":MODEL_ID,"resolved_revision":getattr(model.config,"_commit_hash",None),\n        "device":device,"dtype":str(dtype),"cache_dir":kwargs["cache_dir"],"cache_recovered":bool(attempt)}\n\n\ndef predict_image(predictor, rgb, prompt, threshold=.4):\n    import torch\n    from PIL import Image\n    rgb = np.asarray(rgb)\n    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:\n        raise ValueError("Expected a uint8 RGB image")\n    if not isinstance(prompt,str) or not prompt.strip() or not 0 <= threshold <= 1:\n        raise ValueError("Use a nonempty concept prompt and threshold inside [0,1]")\n    # BatchFeature casts floating image tensors only, preserving integer token IDs.\n    inputs = predictor.processor(images=Image.fromarray(rgb),text=prompt,return_tensors="pt").to(predictor.device,predictor.dtype)\n    with torch.inference_mode():\n        output = predictor.model(**inputs)\n        result = predictor.processor.post_process_instance_segmentation(output,threshold=threshold,\n            mask_threshold=.5,target_sizes=inputs["original_sizes"].tolist())[0]\n    if len(result["masks"]) != len(result["scores"]): raise ValueError("Inconsistent SAM 3 masks and scores")\n    annotations = []\n    for mask,score in zip(result["masks"],result["scores"]):\n        mask = mask.detach().cpu().numpy()\n        score = float(score)\n        if mask.shape != rgb.shape[:2]:\n            raise ValueError(f"SAM 3 mask shape {mask.shape} does not match image {rgb.shape[:2]}")\n        if not np.isfinite(score): raise ValueError(f"SAM 3 returned a nonfinite instance score: {score}")\n        # Transformers 5.2 thresholds masks then casts them to torch.long (0/1),\n        # not torch.bool. Validate values before converting; never threshold logits here.\n        if mask.dtype.kind not in "biuf" or not ((mask == 0) | (mask == 1)).all():\n            raise ValueError(f"SAM 3 returned non-binary mask values (dtype={mask.dtype}); expected only 0/1")\n        annotations.append({"segmentation":mask.astype(bool,copy=True),"detector_score":score,"prompt_label":prompt})\n    return annotations\n', 'roboqc/industrial/review_bundle.py': '"""Image/archive I/O for human-reviewed segmentation proposals. No model execution."""\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport stat\nimport uuid\nimport zipfile\n\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}\n\n\ndef image_files(root):\n    root = Path(root).resolve()\n    if not root.is_dir():\n        raise NotADirectoryError(root)\n    paths = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)\n    if any(not p.resolve().is_relative_to(root) for p in paths):\n        raise ValueError("Image symlink escapes input folder")\n    if not paths:\n        raise ValueError(f"No supported images found recursively in {root}")\n    return paths\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024*1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef pack_images(root, destination):\n    """Package existing photos only, preserving relative directories and original bytes."""\n    root, destination = Path(root).resolve(), Path(destination)\n    paths = image_files(root)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in paths:\n            archive.write(path, path.relative_to(root).as_posix())\n    return {"images": len(paths), "bytes": destination.stat().st_size, "sha256": sha256(destination)}\n\n\ndef extract_images(archive_path, destination, maximum_bytes=10*1024**3, maximum_files=50000):\n    """Extract only images into a fresh directory; preflight ZIP traversal and size limits."""\n    destination = Path(destination)\n    with zipfile.ZipFile(archive_path) as archive:\n        members = []\n        names = set()\n        for member in archive.infolist():\n            name = PurePosixPath(member.filename.replace("\\\\", "/"))\n            if name.is_absolute() or ".." in name.parts or any(":" in part for part in name.parts):\n                raise ValueError(f"Unsafe ZIP member: {member.filename}")\n            if stat.S_ISLNK(member.external_attr >> 16):\n                raise ValueError("ZIP symlinks are not supported")\n            if member.is_dir() or name.suffix.lower() not in IMAGE_SUFFIXES:\n                continue\n            key = name.as_posix().casefold()\n            if key in names:\n                raise ValueError("Duplicate ZIP image path")\n            names.add(key)\n            members.append((member, name))\n        if not members or len(members) > maximum_files or sum(m.file_size for m, _ in members) > maximum_bytes:\n            raise ValueError("Empty image ZIP or configured extraction limits exceeded")\n        destination.mkdir(parents=True, exist_ok=False)\n        for member, name in members:\n            target = destination.joinpath(*name.parts)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(member) as source, target.open("xb") as output:\n                shutil.copyfileobj(source, output)\n    return destination\n\n\ndef register_images(source, bundle):\n    """Save originals and EXIF-normalized RGB PNGs once; proposals reference these IDs."""\n    from PIL import Image, ImageOps\n    source, bundle = Path(source).resolve(), Path(bundle)\n    rows = []\n    for path in image_files(source):\n        relative = path.relative_to(source).as_posix()\n        digest = sha256(path)\n        image_id = hashlib.sha256((relative + "\\0" + digest).encode()).hexdigest()[:24]\n        folder = bundle / "images" / image_id\n        folder.mkdir(parents=True, exist_ok=False)\n        original = folder / ("original" + path.suffix.lower())\n        shutil.copy2(path, original)\n        row = {"image_id": image_id, "source_relative_path": relative, "sha256": digest,\n               "original": original.relative_to(bundle).as_posix(), "status": "ok"}\n        try:\n            with Image.open(path) as opened:\n                rgb = ImageOps.exif_transpose(opened).convert("RGB")\n                rgb.save(folder / "rgb.png")\n                row.update(width=rgb.width, height=rgb.height, rgb=(folder / "rgb.png").relative_to(bundle).as_posix())\n        except (OSError, ValueError) as exc:\n            row.update(status="invalid_image", error=f"{type(exc).__name__}: {exc}")\n        rows.append(row)\n        write_json(bundle / "images.json", rows)\n    reviews = {row["image_id"]: {"status": "unreviewed", "selected_proposal_id": None,\n               "instances": [], "split_group": None} for row in rows}\n    write_json(bundle / "reviews.json", reviews)\n    write_json(bundle / "registration_complete.json", {"images": len(rows)})\n    return rows\n\n\ndef archive_bundle(bundle, destination):\n    """Create a new, CRC-checked snapshot with hashes of every bundled file."""\n    bundle, destination = Path(bundle).resolve(), Path(destination).resolve()\n    if destination.is_relative_to(bundle):\n        raise ValueError("Archive must be outside the bundle")\n    inventory = {p.relative_to(bundle).as_posix(): sha256(p) for p in sorted(bundle.rglob("*"))\n                 if p.is_file() and p.name != "checksums.json"}\n    write_json(bundle / "checksums.json", inventory)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:\n        for path in sorted(bundle.rglob("*")):\n            if path.is_file():\n                archive.write(path, path.relative_to(bundle).as_posix())\n    with zipfile.ZipFile(destination) as archive:\n        if archive.testzip() is not None:\n            raise OSError("Export ZIP failed its CRC check")\n    return {"path": str(destination), "sha256": sha256(destination), "files": len(inventory)+1}\n'}
RUNTIME = Path('/content/roboqc_review_runtime')
for relative,source in SOURCES.items():
    target = RUNTIME / relative; target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source,encoding='utf-8')
sys.path.insert(0,str(RUNTIME))
from roboqc.industrial.review_bundle import image_files, extract_images, register_images, write_json, archive_bundle
from roboqc.industrial.pretrained_compare import DEFAULT_METHODS, PretrainedMethod, crop_image, restore_masks
from roboqc.industrial.sam3 import load_predictor, predict_image
from roboqc.industrial.sam_masks import save_candidates
selected = list(dict.fromkeys(METHODS))
if not selected or any(m not in DEFAULT_METHODS + ['sam3_text'] for m in selected): raise ValueError('Unknown/empty METHODS')
if not PROMPTS or any(not isinstance(p,str) or not p.strip() for p in PROMPTS): raise ValueError('Empty prompt')
if MAX_IMAGES is not None and (not isinstance(MAX_IMAGES,int) or MAX_IMAGES < 1): raise ValueError('MAX_IMAGES must be positive or None')
if MIN_MASK_AREA < 1 or not 0 < MAX_MASK_FRACTION <= 1 or not 0 <= SAM3_THRESHOLD <= 1:
    raise ValueError('Invalid mask filters/confidence')
# Other thresholds match 04c. Edit here before starting a new bundle.
cfg = {'cache':str(CACHE),'sam2_cache':str(CACHE / 'sam2'),'revisions':MODEL_REVISIONS,
    'sam2_id':SAM2_ID,'yoloe_checkpoint':YOLOE_CHECKPOINT,'prompts':PROMPTS,'imgsz':IMGSZ,
    'yoloe_threshold':.25,'dino_threshold':.25,'dino_text_threshold':.20,'owl_threshold':.10,
    'box_nms':.70,'max_detections':50,'points_per_side':32,'points_per_batch':8,
    'crop_layers':0,'sam_iou':.80,'sam_stability':.95,'florence_max_tokens':1024}
settings = {'schema_version':'roboqc-review-1.0','methods':selected,'prompts':PROMPTS,
    'roi_norm':ROI_NORM,'sam3_threshold':SAM3_THRESHOLD,'minimum_area':MIN_MASK_AREA,
    'maximum_fraction':MAX_MASK_FRACTION,'sam2_commit':SAM2_COMMIT,'seed':SEED,
    'max_images':MAX_IMAGES,'config':cfg,'weights_in_export':False,'accuracy':'not_measured',
    'runtime_sha256':{name:hashlib.sha256(source.encode()).hexdigest() for name,source in SOURCES.items()}}
if RESUME_BUNDLE:
    OUT = Path(RESUME_BUNDLE)
    if not (OUT / 'registration_complete.json').exists():
        raise ValueError('Input registration was interrupted. Start a new bundle from the original input ZIP.')
    previous = json.loads((OUT / 'settings.json').read_text())
    if previous != settings: raise ValueError('Configuration differs from saved bundle. Restore it or start a new bundle.')
    samples = json.loads((OUT / 'images.json').read_text())
    print('Resuming from saved originals; INPUT_PATH is ignored.')
else:
    if INPUT_PATH:
        source = Path(INPUT_PATH)
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1: raise ValueError('Upload exactly one input ZIP')
        name,blob = next(iter(uploaded.items()))
        if Path(name).suffix.lower() != '.zip': raise ValueError('Upload a ZIP of the /data folder')
        source = Path(ROOT) / 'inputs' / (uuid.uuid4().hex + '_' + Path(name).name)
        source.parent.mkdir(parents=True,exist_ok=True)
        with source.open('xb') as handle: handle.write(blob)
        del uploaded, blob
        print('Input saved in Drive; reuse INPUT_PATH =',repr(str(source)))
    if not source.exists(): raise FileNotFoundError(source)
    if source.is_file():
        local_zip = Path('/content') / ('review_input_' + uuid.uuid4().hex + '.zip')
        shutil.copy2(source,local_zip)
        source = extract_images(local_zip,Path('/content') / ('review_images_' + uuid.uuid4().hex),
            maximum_bytes=int(MAX_EXTRACT_GB*1024**3),maximum_files=MAX_EXTRACT_FILES)
    discovered = image_files(source)
    OUT = RUN / ('review_bundle_' + uuid.uuid4().hex); OUT.mkdir()
    write_json(OUT / 'settings.json',settings)
    # Register ALL input photos, even when MAX_IMAGES limits inference.
    samples = register_images(source,OUT)
    print('Recursively discovered:',len(discovered),'| invalid:',sum(s['status'] != 'ok' for s in samples))
active = [s for s in samples if s['status'] == 'ok']
if MAX_IMAGES is not None: active = active[:MAX_IMAGES]
if not active: raise ValueError('No decodable images')
print('Bundle:',OUT,'| images to process:',len(active),'| models:',selected)
print('For recovery, set RESUME_BUNDLE =',repr(str(OUT)))


## Generate proposals (long-running; execute manually)

Each SAM 3 prompt is a separate alternative. The other methods use the
configured prompt list together (SAM automatic masks ignore text). Never
union alternatives blindly: they may duplicate objects or segment subparts.
No confidence-based cross-model ranking is used. Every completed prediction
is saved immediately, including zero detections. Errors remain explicit.
Rerun with `RESUME_BUNDLE` and identical settings to skip successful proposals
and retry failures into fresh folders. Existing masks/review edits survive.


In [ ]:
index_path = OUT / 'proposals.json'
records = json.loads(index_path.read_text()) if index_path.exists() else []
completed = {r['proposal_id'] for r in records if r['status'] == 'ok'}
for method in selected:
    variants = list(enumerate(PROMPTS)) if method == 'sam3_text' else [(0,None)]
    pending = [(sample,pi,prompt) for sample in active for pi,prompt in variants
               if f"{sample['image_id']}:{method}:{pi}" not in completed]
    if not pending: continue
    runner = None
    try:
        started = time.perf_counter()
        if method == 'sam3_text':
            runner,provenance = load_predictor(CACHE / 'sam3',MODEL_REVISIONS.get('facebook/sam3','main'),DEVICE)
        else:
            runner = PretrainedMethod(method,cfg,DEVICE)
            provenance = runner.provenance
        load_seconds = time.perf_counter()-started
        load_error = None
    except Exception:
        load_error = traceback.format_exc(); print(load_error)
    try:
        for sample,pi,prompt in pending:
            proposal_id = f"{sample['image_id']}:{method}:{pi}"
            attempt = OUT / 'proposals' / sample['image_id'] / method / (f'prompt_{pi:02d}_' + uuid.uuid4().hex)
            attempt.mkdir(parents=True)
            row = {'proposal_id':proposal_id,'image_id':sample['image_id'],'method':method,
                   'prompt':prompt,'prompts':PROMPTS,'path':attempt.relative_to(OUT).as_posix(),
                   'status':'error','device':DEVICE,'accuracy':'not_measured'}
            try:
                if load_error: raise RuntimeError('Model load failed; see traceback below.\n' + load_error)
                write_json(attempt / 'provenance.json',provenance)
                with Image.open(OUT / sample['rgb']) as opened: rgb = np.asarray(opened.convert('RGB'))
                crop,roi = crop_image(rgb,ROI_NORM)
                if GPU: torch.cuda.synchronize()
                started = time.perf_counter()
                if method == 'sam3_text':
                    annotations = predict_image(runner,crop,prompt,SAM3_THRESHOLD)
                    details = {'score_type':'SAM 3 confidence; not observed accuracy'}
                else:
                    annotations,details = runner.predict(crop)
                if GPU: torch.cuda.synchronize()
                seconds = time.perf_counter()-started
                annotations = restore_masks(annotations,rgb.shape,roi)
                fraction = MAX_MASK_FRACTION*crop.shape[0]*crop.shape[1]/(rgb.shape[0]*rgb.shape[1])
                result,_ = save_candidates(rgb,annotations,attempt / 'masks',MIN_MASK_AREA,fraction,SEED)
                row.update(status='ok',elapsed_s=seconds,load_seconds_including_downloads=load_seconds,
                    raw_masks=result['raw_masks'],kept_masks=result['kept_candidates'],roi=list(roi),details=details,
                    candidates=(attempt / 'masks/candidates.json').relative_to(OUT).as_posix(),
                    overlay=(attempt / 'masks/overlay.png').relative_to(OUT).as_posix())
                completed.add(proposal_id)
            except Exception as exc:
                row['error'] = f'{type(exc).__name__}: {exc}'
                (attempt / 'error.txt').write_text(traceback.format_exc(),encoding='utf-8')
                traceback.print_exc(file=sys.stdout)
            write_json(attempt / 'result.json',row)
            # Index the latest attempt; old attempt directories are deliberately preserved.
            records = [r for r in records if r['proposal_id'] != proposal_id] + [row]
            write_json(index_path,records)
            pd.json_normalize(records).to_csv(OUT / 'summary.csv',index=False)
            print(sample['source_relative_path'],method,prompt,row['status'],row.get('kept_masks'),flush=True)
    finally:
        runner = None
        gc.collect()
        if GPU: torch.cuda.empty_cache()
display(pd.DataFrame(records).groupby(['method','status']).size().rename('proposals').reset_index())


In [ ]:
import matplotlib.pyplot as plt
for sample in active[:PREVIEW_IMAGES]:
    choices = [r for r in records if r['image_id'] == sample['image_id']]
    fig,axes = plt.subplots((len(choices)+3)//3,3,figsize=(18,6*((len(choices)+3)//3)),squeeze=False)
    for axis in axes.flat: axis.axis('off')
    with Image.open(OUT / sample['rgb']) as view: axes.flat[0].imshow(view)
    axes.flat[0].set_title(sample['source_relative_path'])
    for axis,row in zip(list(axes.flat)[1:],choices):
        axis.set_title(row['method'] + (' / ' + row['prompt'] if row['prompt'] else ''))
        if row['status'] == 'ok':
            with Image.open(OUT / row['overlay']) as view: axis.imshow(view)
        else: axis.text(.02,.5,'FAILED — see error.txt',wrap=True)
    plt.tight_layout(); plt.show()
print('Confidence scales differ. No reviewed truth: no accuracy, mAP or automatic winner.')


## Export to Drive / contract for the later review app

`images.json`: stable image IDs, nested source paths, dimensions, original
hashes, original bytes and EXIF-normalized RGB paths. All proposal masks
use the normalized RGB coordinates, never the unnormalized original.

`proposals.json`: latest outcome per image/model/prompt, provenance, timings,
errors and relative paths. Per-proposal `raw_masks.npz` contains ALL boolean
instances, including filtered ones; `candidates.json` maps raw indices to
PNGs, bounding boxes and scores. Separate masks preserve holes and overlaps.
Empty successful predictions differ from failed methods and unprocessed images.

`reviews.json`: ONE future annotation record per image, initially unreviewed.
The app should choose a proposal, allow adding/deleting/splitting/merging and
painting masks, assign class IDs, then save each accepted instance as a
separate binary PNG in `reviewed/<image_id>/`. Set `status='accepted'`,
`selected_proposal_id`, `split_group`, and `instances=[{'instance_id':1,
'class_id':0,'mask_file':'reviewed/.../instance_1.png'}]`.
One annotation set per image does NOT mean one merged mask for all pieces.
An accepted empty scene must be explicit; unreviewed empty lists are NOT negatives.

Later: export reviewed labels to COCO masks or YOLO segmentation, train a
compact segmentation model and evaluate on untouched reviewed holdouts.
YOLO polygon export needs an explicit policy for holes/disconnected masks;
retain these lossless PNG/NPZ masters. Split by capture session/scene BEFORE
augmentation; near-identical frames and their derivatives stay in one split.
Use joint image/mask geometry transforms, image-only lighting/color changes,
and reviewed-mask cut/paste onto new backgrounds. Do not change defect/color
labels through augmentation, invent hidden surfaces or treat synthetic
shadows as real defect evidence. Validate on real, unaugmented test images.

This notebook prepares the archive contract, NOT the annotation app or
training/augmentation implementation. Human review remains required before training.
Check model licenses, especially before redistribution or deployment.


In [ ]:
"""Paste this whole file into 04e's LAST cell. No model imports or inference."""

import base64
from collections import deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
import hashlib
import io
import json
import os
import shutil
import tempfile
from pathlib import Path, PurePosixPath
import stat
import uuid
import zipfile
from tqdm.auto import tqdm


def list_export_files(bundle, cache_dir, workers=4, refresh=False):
    """One stat per entry, parallel folders, resumable inventory of FROZEN outputs.

    Directory mtimes catch additions/removals; root manifests guard normal edits.
    Use refresh=True after manually editing files in-place, without updating manifests.
    """
    bundle,cache_dir = Path(bundle).resolve(),Path(cache_dir).resolve()
    if not isinstance(workers,int) or not 1 <= workers <= 16:
        raise ValueError('Listing workers must be an integer from 1 to 16.')
    if cache_dir.is_relative_to(bundle): raise ValueError('Inventory cache must be outside the bundle.')
    cache_dir.mkdir(parents=True,exist_ok=True)

    def scan(relative,cached=None):
        folder = bundle / relative
        before = os.stat(folder,follow_symlinks=False)
        if not stat.S_ISDIR(before.st_mode): raise ValueError('Directory symlink or unsupported source folder.')
        if cached and cached['mtime_ns']==before.st_mtime_ns: return cached,True
        files,children = {},[]
        with os.scandir(folder) as entries:
            for entry in entries:
                info = entry.stat(follow_symlinks=False)  # DirEntry: one metadata lookup, no resolve per file.
                if stat.S_ISLNK(info.st_mode): raise ValueError('Source symlinks are not supported in the export inventory.')
                if stat.S_ISDIR(info.st_mode): children.append(entry.name)
                elif stat.S_ISREG(info.st_mode): files[entry.name]=[info.st_size,info.st_mtime_ns]
        after = os.stat(folder,follow_symlinks=False)
        if before.st_mtime_ns!=after.st_mtime_ns:
            raise RuntimeError('Source folder changed while listing. Stop concurrent inference/review writes.')
        return {'directory':relative,'mtime_ns':after.st_mtime_ns,'files':files,'directories':sorted(children)},False

    root,_ = scan('')  # Cheap root-manifest guard, even on cached runs.
    signature = hashlib.sha256(json.dumps([str(bundle),root['files']],sort_keys=True).encode()).hexdigest()[:16]
    journal = cache_dir / (bundle.name + '_' + signature + '.inventory.jsonl')
    cache = {}; ignored=0
    if journal.exists() and not refresh:
        with journal.open(encoding='utf-8') as handle:
            for line in handle:
                if not line.strip(): continue
                try:
                    row = json.loads(line)
                    if not isinstance(row,dict): raise ValueError('Invalid inventory record')
                    if row.get('reset_inventory') is True:
                        cache.clear()
                        continue
                    if not isinstance(row.get('files'),dict) or not isinstance(row.get('directories'),list):
                        raise ValueError('Invalid inventory record')
                    relative = PurePosixPath(row['directory'])
                    names = [*row['files'],*row['directories']]
                    if (relative.is_absolute() or '..' in relative.parts or '\\' in row['directory']
                            or any(':' in part for part in relative.parts)
                            or any(not n or n in {'.','..'} or '/' in n or '\\' in n for n in names)):
                        raise ValueError('Unsafe inventory path')
                    if not isinstance(row['mtime_ns'],int): raise ValueError('Invalid directory timestamp')
                    cache[row['directory']]=row
                except (ValueError,KeyError,TypeError): ignored+=1
    if ignored: print('Ignored incomplete inventory records:',ignored,'; affected folders will be rescanned.')
    print('Inventory cache:',journal)
    print('Outputs must stay unchanged; set REFRESH_EXPORT_INVENTORY=True after manual mask edits.')
    result={}; reused=0; fresh=0; queue=deque(); running={}
    # Leading newlines separate new records from a torn last write after a disconnect.
    with journal.open('a',encoding='utf-8') as handle, tqdm(desc=f'List saved files ({workers} workers)',unit='file') as bar:
        if refresh:
            # A durable reset lets the next run resume even an interrupted forced refresh.
            handle.write('\n' + json.dumps({'reset_inventory':True}))
            handle.flush()
        def accept(row,from_cache):
            nonlocal reused,fresh
            relative=row['directory']
            for name,metadata in row['files'].items():
                result[(PurePosixPath(relative) / name).as_posix()]=metadata
            queue.extend((PurePosixPath(relative) / name).as_posix() for name in row['directories'])
            if from_cache: reused+=1
            else:
                fresh+=1
                handle.write('\n' + json.dumps(row,ensure_ascii=False))
                if fresh % 16==0: handle.flush()
            bar.update(len(row['files']))
            bar.set_postfix(cached_dirs=reused,scanned_dirs=fresh,queued_dirs=len(queue)+len(running))
        accept(root,cache.get('')==root)
        with ThreadPoolExecutor(max_workers=workers) as pool:
            while queue or running:
                while queue and len(running)<workers:
                    relative=queue.popleft()
                    running[pool.submit(scan,relative,cache.get(relative))]=relative
                finished,_ = wait(running,return_when=FIRST_COMPLETED)
                for future in finished:
                    running.pop(future)
                    row,from_cache=future.result()
                    accept(row,from_cache)
    print('Inventory:',len(result),'files | reused directories:',reused,'| scanned directories:',fresh)
    return result


def export_saved_bundle(bundle, export_dir, batch_size=1000, search_dirs=(),
                        listing_workers=4, refresh_inventory=False,
                        fast_local=True, read_workers=6, checkpoint_mb=256):
    bundle, export_dir = Path(bundle).resolve(), Path(export_dir).resolve()
    if not (bundle / 'images.json').is_file():
        raise ValueError('Set BUNDLE_TO_EXPORT to your saved review_bundle folder.')
    if export_dir.is_relative_to(bundle):
        raise ValueError('Export folder must be outside the input bundle.')
    export_dir.mkdir(parents=True, exist_ok=True)
    records_path = bundle / 'proposals.json'
    records = json.loads(records_path.read_text()) if records_path.exists() else []
    completed = {r['proposal_id'] for r in records if r['status'] == 'ok'}
    print('Saved successful proposals:', len(completed))
    if (bundle / 'settings.json').exists():
        settings = json.loads((bundle / 'settings.json').read_text())
        images = [s for s in json.loads((bundle / 'images.json').read_text()) if s['status'] == 'ok']
        if settings['max_images'] is not None: images = images[:settings['max_images']]
        expected = {f"{s['image_id']}:{m}:{i}" for s in images for m in settings['methods']
                    for i in (range(len(settings['prompts'])) if m == 'sam3_text' else [0])}
        print('Expected:', len(expected), '| missing/failed:', len(expected-completed))
    print('Export only: no inference or processing of images will run.')

    files = list_export_files(bundle,export_dir,workers=listing_workers,refresh=refresh_inventory)
    fingerprint = hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()

    def checked_members(target):
        # Read only the ZIP directory. Payload CRC/SHA is NOT re-verified here.
        with zipfile.ZipFile(target) as archive:
            members = archive.infolist()
        result = {m.filename: m.file_size for m in members}
        if len(result) != len(members) or any(n not in files or size != files[n][0] for n, size in result.items()):
            raise ValueError('Archive entries differ from this snapshot (names/sizes/duplicates).')
        return result

    attempts = set()
    extra_roots = {Path(p).resolve() for p in search_dirs}
    roots = {export_dir, bundle.parent, *extra_roots}
    with tqdm(desc='Find export attempts', unit='attempt') as bar:
        for root in sorted(roots):
            for directory, children, names in os.walk(root):
                children[:] = [n for n in children if not n.startswith('review_bundle_')
                               and n not in {'images', 'proposals', 'runtime', 'inputs'}]
                for name in names:
                    if ((name.startswith(bundle.name) and ('.zip' in name or name.endswith('.export.json')))
                            or (root in extra_roots and name == 'results.zip')):
                        path = Path(directory) / name
                        if path not in attempts: attempts.add(path); bar.update(1)
    rows = []
    covered = set()
    # Checkpoint tails can be inspected without touching an interrupted work ZIP.
    for path in tqdm(sorted(p for p in attempts if p.name.endswith('.export.json')), desc='Review checkpoints', unit='attempt'):
        row = {'path':str(path), 'kind':'checkpoint', 'bytes':0, 'files':0, 'usable':False}
        try:
            state = json.loads(path.read_text())
            work = Path(state['archive']).resolve()
            covered.add(work)
            if (state['bundle'] != str(bundle) or state['fingerprint'] != fingerprint
                    or work.parent != export_dir or not work.name.startswith(bundle.name + '_resume_')
                    or not work.name.endswith('.work.zip')):
                raise ValueError('Checkpoint is for a different snapshot or unsupported work path.')
            offset = state['directory_offset']
            if not isinstance(offset,int) or offset < 0 or work.stat().st_size < offset:
                raise ValueError('Committed ZIP prefix is missing.')
            members = checked_members(io.BytesIO(base64.b64decode(state['tail'],validate=True)))
            row.update(usable=True, bytes=sum(members.values()), files=len(members), state=state)
        except (OSError, ValueError, KeyError, TypeError, zipfile.BadZipFile) as exc:
            row['reason'] = str(exc)
        rows.append(row)
    for path in tqdm(sorted(p for p in attempts if not p.name.endswith('.export.json') and p.resolve() not in covered), desc='Review ZIP attempts', unit='attempt'):
        row = {'path':str(path), 'kind':'archive', 'bytes':0, 'files':0, 'usable':False}
        try:
            members = checked_members(path)
            row.update(usable=True, bytes=sum(members.values()), files=len(members))
        except (OSError, ValueError, zipfile.BadZipFile) as exc:
            row['reason'] = str(exc) + ' (preserved; no usable checkpoint)'
        rows.append(row)
    total = sum(size for size, _ in files.values())
    rows.sort(key=lambda r:(r['usable'],r['bytes'],r['files'],
                           r['kind']=='archive' and r['path'].endswith('.zip') and not r['path'].endswith('.work.zip')), reverse=True)
    print('Export attempts (ZIP directory checks only; not full payload verification):')
    for row in rows:
        pct = 100*row['bytes']/total if total else 0
        print(f"{pct:6.1f}% | {row['files']}/{len(files)} files | {row['kind']} | "
              f"{'recoverable' if row['usable'] else row.get('reason','unusable')} | {row['path']}")
    best = next((row for row in rows if row['usable']),None)
    if best:
        print('Most export progress:',best['path'])
        candidate = Path(best['path'])
        if best['kind']=='archive' and best['files']==len(files) and candidate.suffix=='.zip' and not candidate.name.endswith('.work.zip'):
            print('Existing complete ZIP (file list/sizes checked):',candidate)
            return candidate

    state_path = (Path(best['path']) if best and best['kind']=='checkpoint' else
                  export_dir / (bundle.name + '_' + fingerprint[:16] + '_' + uuid.uuid4().hex[:8] + '.export.json'))

    def save_checkpoint(work, directory_offset):
        # Back up the committed central directory BEFORE the next append overwrites it.
        # Restoring this tail discards only uncommitted bytes in this owned work ZIP.
        with work.open('rb') as handle:
            handle.seek(directory_offset)
            tail = base64.b64encode(handle.read()).decode('ascii')
        state = {'bundle': str(bundle), 'fingerprint': fingerprint, 'archive': str(work),
                 'directory_offset': directory_offset, 'tail': tail}
        temporary = state_path.with_name(state_path.name + '.' + uuid.uuid4().hex + '.tmp')
        temporary.write_text(json.dumps(state), encoding='utf-8')
        temporary.replace(state_path)

    if best and best['kind']=='checkpoint':
        state = best['state']
        work = Path(state['archive']).resolve()
        if (state['bundle'] != str(bundle) or state['fingerprint'] != fingerprint
                or work.parent != export_dir or not work.name.startswith(bundle.name + '_resume_')
                or not work.name.endswith('.work.zip')):
            raise ValueError('Export checkpoint does not match this bundle; existing files were preserved.')
        offset = state['directory_offset']
        tail = base64.b64decode(state['tail'], validate=True)
        if not isinstance(offset, int) or offset < 0 or work.stat().st_size < offset:
            raise ValueError('Committed ZIP bytes are missing. Existing files were preserved.')
        with work.open('r+b') as handle:
            handle.seek(offset)
            handle.write(tail)
            handle.truncate(offset + len(tail))
        print('Resuming saved export:', work)
    else:
        work = export_dir / (bundle.name + '_resume_' + uuid.uuid4().hex + '.work.zip')
        if best:
            # Clone a readable legacy partial ZIP. Never append to the original attempt.
            source_path = Path(best['path'])
            with source_path.open('rb') as source, work.open('xb') as destination, tqdm(
                    total=source_path.stat().st_size,desc='Preserve previous ZIP progress',unit='B',unit_scale=True,unit_divisor=1024) as bar:
                for chunk in iter(lambda:source.read(4*1024*1024),b''):
                    destination.write(chunk); bar.update(len(chunk))
            with zipfile.ZipFile(work) as archive: pass
        else:
            with zipfile.ZipFile(work, 'x', compression=zipfile.ZIP_STORED) as archive: pass
        save_checkpoint(work, archive.start_dir)
        print('New resumable export:', work)

    with zipfile.ZipFile(work) as archive:
        committed = {m.filename: m.file_size for m in archive.infolist()}
    if any(name not in files or size != files[name][0] for name, size in committed.items()):
        raise ValueError('Checkpoint contains unexpected files. Existing source data were preserved.')
    pending = sorted(name for name in files if name not in committed)
    print('Files already exported:', len(committed), '| files remaining:', len(pending))

    # Fast mode: work on Colab's SSD, not the Google Drive FUSE mount.
    # The original Drive checkpoint remains recoverable while local changes are in progress.
    # At each checkpoint, transfer ONLY the ZIP bytes changed since the last checkpoint,
    # then atomically update the Drive-side checkpoint JSON.
    if fast_local and (not isinstance(read_workers, int) or not 1 <= read_workers <= 16):
        raise ValueError('read_workers must be an integer from 1 to 16.')
    if fast_local and checkpoint_mb <= 0:
        raise ValueError('checkpoint_mb must be positive.')

    def read_prefetched(names):
        """Ordered, bounded parallel Drive reads; no more than 2*workers buffered tasks.

        Large files are streamed by the writer to avoid holding them all in RAM.
        Each source is checked against the frozen inventory before and after reading.
        """
        max_buffer_bytes = 16 * 1024 * 1024

        def read_one(name):
            path = bundle / name
            before = path.stat()
            if [before.st_size, before.st_mtime_ns] != files[name]:
                raise RuntimeError(f'Source changed before export: {name}')
            if before.st_size > max_buffer_bytes:
                return name, None
            data = path.read_bytes()
            after = path.stat()
            if len(data) != files[name][0] or [after.st_size, after.st_mtime_ns] != files[name]:
                raise RuntimeError(f'Source changed while reading: {name}')
            return name, data

        with ThreadPoolExecutor(max_workers=read_workers) as pool:
            todo = iter(names)
            future_queue = deque()
            for _ in range(read_workers * 2):
                name = next(todo, None)
                if name is None:
                    break
                future_queue.append(pool.submit(read_one, name))
            while future_queue:
                item = future_queue.popleft().result()
                name = next(todo, None)
                if name is not None:
                    future_queue.append(pool.submit(read_one, name))
                yield item

    def write_remaining(target, *, local_mode=False, initial_offset=None):
        last_synced_offset = initial_offset
        unsynced_bytes = 0
        threshold = int(checkpoint_mb * 1024 * 1024)

        def sync_drive(new_offset):
            nonlocal last_synced_offset, unsynced_bytes
            if last_synced_offset is None:
                raise RuntimeError('Missing last committed ZIP offset.')
            # Keep existing checkpoint JSON UNCHANGED until the new ZIP suffix
            # is fully copied. A failed copy can be rolled back using its tail.
            with target.open('rb') as source, work.open('r+b') as dest:
                source.seek(last_synced_offset)
                dest.seek(last_synced_offset)
                for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''):
                    dest.write(chunk)
                dest.truncate(target.stat().st_size)
                dest.flush()
            save_checkpoint(work, new_offset)
            last_synced_offset = new_offset
            unsynced_bytes = 0

        # ZIP_STORED is deliberately retained: same file names and original
        # bytes, no lossy conversion or additional compression work.
        with tqdm(total=total, initial=sum(committed.values()), desc='Export saved files',
                  unit='B', unit_scale=True, unit_divisor=1024) as bar:
            items = (read_prefetched(pending) if local_mode else ((name, None) for name in pending))
            for start in range(0, len(pending), batch_size):
                batch = pending[start:start + batch_size]
                with zipfile.ZipFile(target, 'a', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
                    for _ in batch:
                        name, payload = next(items)
                        if payload is not None:
                            archive.writestr(name, payload, compress_type=zipfile.ZIP_STORED)
                            bar.update(len(payload))
                        else:
                            path = bundle / name
                            before = path.stat()
                            if [before.st_size, before.st_mtime_ns] != files[name]:
                                raise RuntimeError(f'Source changed before export: {name}')
                            written = 0
                            with path.open('rb') as source, archive.open(name, 'w', force_zip64=True) as dest:
                                for chunk in iter(lambda: source.read(4 * 1024 * 1024), b''):
                                    dest.write(chunk)
                                    written += len(chunk)
                                    bar.update(len(chunk))
                            after = path.stat()
                            if written != files[name][0] or [after.st_size, after.st_mtime_ns] != files[name]:
                                raise RuntimeError(f'Source changed during export: {name}')
                        unsynced_bytes += files[name][0]
                if not local_mode:
                    save_checkpoint(work, archive.start_dir)
                    bar.set_postfix(files_committed=len(committed) + min(start + batch_size, len(pending)))
                elif unsynced_bytes >= threshold or start + batch_size >= len(pending):
                    sync_drive(archive.start_dir)
                    bar.set_postfix(files_committed=len(committed) + min(start + batch_size, len(pending)))
            if not pending and local_mode:
                # No new files. Existing saved checkpoint still describes the ZIP.
                pass

    if fast_local and pending:
        local_root = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir())
        space_needed = total + max(512 * 1024 * 1024, total // 10)
        space_available = shutil.disk_usage(local_root).free
        if space_available < space_needed:
            raise RuntimeError(f'Not enough Colab local disk for safe fast export: '
                               f'need {space_needed / 2**30:.1f} GiB free, '
                               f'have {space_available / 2**30:.1f} GiB. '
                               'Clear local disk or call with fast_local=False.')
        with tempfile.TemporaryDirectory(prefix='roboqc_export_', dir=local_root) as scratch:
            local_work = Path(scratch) / work.name
            print('Copying the resumable ZIP to Colab local disk:', work.stat().st_size, 'bytes')
            with work.open('rb') as original, local_work.open('wb') as destination:
                shutil.copyfileobj(original, destination, length=8 * 1024 * 1024)
            last_offset = json.loads(state_path.read_text())['directory_offset']
            print(f'Fast export: {read_workers} parallel Drive readers, '
                  f'{batch_size} files per SSD batch, Drive checkpoint about every {checkpoint_mb} MiB.')
            write_remaining(local_work, local_mode=True, initial_offset=last_offset)
    else:
        if not fast_local:
            print('Drive-direct fallback; consider fast_local=True for much lower Drive write overhead.')
        write_remaining(work)

    if len(checked_members(work)) != len(files):
        raise RuntimeError('Final ZIP file list/size check failed; work ZIP and checkpoints preserved.')
    final = work.with_name(work.name.removesuffix('.work.zip') + '.zip')
    if final.exists():
        raise FileExistsError(final)
    work.rename(final)
    print('Drive ZIP ready:', final)
    print('File list/sizes checked; full payload CRC/SHA re-verification skipped for speed.')
    return final


if __name__ == '__main__':
    # Can be executed alone after a runtime reset: uses the known saved bundle path.
    # Set these globals above, if desired, to tune performance:
    EXPORT_FAST_LOCAL = globals().get('EXPORT_FAST_LOCAL', True)
    EXPORT_READ_WORKERS = globals().get('EXPORT_READ_WORKERS', 6)
    EXPORT_CHECKPOINT_MB = globals().get('EXPORT_CHECKPOINT_MB', 256)
    DEFAULT_BUNDLE_TO_EXPORT = ''  # Optional exact saved bundle path after a runtime reset
    BUNDLE_TO_EXPORT = globals().get('OUT') or globals().get('RESUME_BUNDLE') or DEFAULT_BUNDLE_TO_EXPORT
    EXPORT_FOLDER = Path(globals().get('ROOT', '/content/drive/MyDrive/Robotekin2026')) / 'exports'
    if not BUNDLE_TO_EXPORT or str(BUNDLE_TO_EXPORT) == 'auto':
        raise ValueError('Set BUNDLE_TO_EXPORT above to your exact saved review_bundle folder.')
    if not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    export_saved_bundle(BUNDLE_TO_EXPORT, EXPORT_FOLDER,
        search_dirs=[globals().get('EXPORT_LOCAL_DIR','/content/roboqc_review_exports')],
        listing_workers=globals().get('EXPORT_LISTING_WORKERS',4),
        refresh_inventory=globals().get('REFRESH_EXPORT_INVENTORY',False),
        fast_local=globals().get('EXPORT_FAST_LOCAL',True),
        read_workers=globals().get('EXPORT_READ_WORKERS',6),
        checkpoint_mb=globals().get('EXPORT_CHECKPOINT_MB',256))

In [ ]:
# """Paste this whole file into 04e's LAST cell. No model imports or inference."""

# import base64
# from collections import deque
# from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
# import hashlib
# import io
# import json
# import os
# from pathlib import Path, PurePosixPath
# import stat
# import uuid
# import zipfile
# from tqdm.auto import tqdm


# def list_export_files(bundle, cache_dir, workers=4, refresh=False):
#     """One stat per entry, parallel folders, resumable inventory of FROZEN outputs.

#     Directory mtimes catch additions/removals; root manifests guard normal edits.
#     Use refresh=True after manually editing files in-place, without updating manifests.
#     """
#     bundle,cache_dir = Path(bundle).resolve(),Path(cache_dir).resolve()
#     if not isinstance(workers,int) or not 1 <= workers <= 16:
#         raise ValueError('Listing workers must be an integer from 1 to 16.')
#     if cache_dir.is_relative_to(bundle): raise ValueError('Inventory cache must be outside the bundle.')
#     cache_dir.mkdir(parents=True,exist_ok=True)

#     def scan(relative,cached=None):
#         folder = bundle / relative
#         before = os.stat(folder,follow_symlinks=False)
#         if not stat.S_ISDIR(before.st_mode): raise ValueError('Directory symlink or unsupported source folder.')
#         if cached and cached['mtime_ns']==before.st_mtime_ns: return cached,True
#         files,children = {},[]
#         with os.scandir(folder) as entries:
#             for entry in entries:
#                 info = entry.stat(follow_symlinks=False)  # DirEntry: one metadata lookup, no resolve per file.
#                 if stat.S_ISLNK(info.st_mode): raise ValueError('Source symlinks are not supported in the export inventory.')
#                 if stat.S_ISDIR(info.st_mode): children.append(entry.name)
#                 elif stat.S_ISREG(info.st_mode): files[entry.name]=[info.st_size,info.st_mtime_ns]
#         after = os.stat(folder,follow_symlinks=False)
#         if before.st_mtime_ns!=after.st_mtime_ns:
#             raise RuntimeError('Source folder changed while listing. Stop concurrent inference/review writes.')
#         return {'directory':relative,'mtime_ns':after.st_mtime_ns,'files':files,'directories':sorted(children)},False

#     root,_ = scan('')  # Cheap root-manifest guard, even on cached runs.
#     signature = hashlib.sha256(json.dumps([str(bundle),root['files']],sort_keys=True).encode()).hexdigest()[:16]
#     journal = cache_dir / (bundle.name + '_' + signature + '.inventory.jsonl')
#     cache = {}; ignored=0
#     if journal.exists() and not refresh:
#         with journal.open(encoding='utf-8') as handle:
#             for line in handle:
#                 if not line.strip(): continue
#                 try:
#                     row = json.loads(line)
#                     if not isinstance(row,dict): raise ValueError('Invalid inventory record')
#                     if row.get('reset_inventory') is True:
#                         cache.clear()
#                         continue
#                     if not isinstance(row.get('files'),dict) or not isinstance(row.get('directories'),list):
#                         raise ValueError('Invalid inventory record')
#                     relative = PurePosixPath(row['directory'])
#                     names = [*row['files'],*row['directories']]
#                     if (relative.is_absolute() or '..' in relative.parts or '\\' in row['directory']
#                             or any(':' in part for part in relative.parts)
#                             or any(not n or n in {'.','..'} or '/' in n or '\\' in n for n in names)):
#                         raise ValueError('Unsafe inventory path')
#                     if not isinstance(row['mtime_ns'],int): raise ValueError('Invalid directory timestamp')
#                     cache[row['directory']]=row
#                 except (ValueError,KeyError,TypeError): ignored+=1
#     if ignored: print('Ignored incomplete inventory records:',ignored,'; affected folders will be rescanned.')
#     print('Inventory cache:',journal)
#     print('Outputs must stay unchanged; set REFRESH_EXPORT_INVENTORY=True after manual mask edits.')
#     result={}; reused=0; fresh=0; queue=deque(); running={}
#     # Leading newlines separate new records from a torn last write after a disconnect.
#     with journal.open('a',encoding='utf-8') as handle, tqdm(desc=f'List saved files ({workers} workers)',unit='file') as bar:
#         if refresh:
#             # A durable reset lets the next run resume even an interrupted forced refresh.
#             handle.write('\n' + json.dumps({'reset_inventory':True}))
#             handle.flush()
#         def accept(row,from_cache):
#             nonlocal reused,fresh
#             relative=row['directory']
#             for name,metadata in row['files'].items():
#                 result[(PurePosixPath(relative) / name).as_posix()]=metadata
#             queue.extend((PurePosixPath(relative) / name).as_posix() for name in row['directories'])
#             if from_cache: reused+=1
#             else:
#                 fresh+=1
#                 handle.write('\n' + json.dumps(row,ensure_ascii=False))
#                 if fresh % 16==0: handle.flush()
#             bar.update(len(row['files']))
#             bar.set_postfix(cached_dirs=reused,scanned_dirs=fresh,queued_dirs=len(queue)+len(running))
#         accept(root,cache.get('')==root)
#         with ThreadPoolExecutor(max_workers=workers) as pool:
#             while queue or running:
#                 while queue and len(running)<workers:
#                     relative=queue.popleft()
#                     running[pool.submit(scan,relative,cache.get(relative))]=relative
#                 finished,_ = wait(running,return_when=FIRST_COMPLETED)
#                 for future in finished:
#                     running.pop(future)
#                     row,from_cache=future.result()
#                     accept(row,from_cache)
#     print('Inventory:',len(result),'files | reused directories:',reused,'| scanned directories:',fresh)
#     return result


# def export_saved_bundle(bundle, export_dir, batch_size=50, search_dirs=(), listing_workers=4, refresh_inventory=False):
#     bundle, export_dir = Path(bundle).resolve(), Path(export_dir).resolve()
#     if not (bundle / 'images.json').is_file():
#         raise ValueError('Set BUNDLE_TO_EXPORT to your saved review_bundle folder.')
#     if export_dir.is_relative_to(bundle):
#         raise ValueError('Export folder must be outside the input bundle.')
#     export_dir.mkdir(parents=True, exist_ok=True)
#     records_path = bundle / 'proposals.json'
#     records = json.loads(records_path.read_text()) if records_path.exists() else []
#     completed = {r['proposal_id'] for r in records if r['status'] == 'ok'}
#     print('Saved successful proposals:', len(completed))
#     if (bundle / 'settings.json').exists():
#         settings = json.loads((bundle / 'settings.json').read_text())
#         images = [s for s in json.loads((bundle / 'images.json').read_text()) if s['status'] == 'ok']
#         if settings['max_images'] is not None: images = images[:settings['max_images']]
#         expected = {f"{s['image_id']}:{m}:{i}" for s in images for m in settings['methods']
#                     for i in (range(len(settings['prompts'])) if m == 'sam3_text' else [0])}
#         print('Expected:', len(expected), '| missing/failed:', len(expected-completed))
#     print('Export only: no inference or processing of images will run.')

#     files = list_export_files(bundle,export_dir,workers=listing_workers,refresh=refresh_inventory)
#     fingerprint = hashlib.sha256(json.dumps(files, sort_keys=True).encode()).hexdigest()

#     def checked_members(target):
#         # Read only the ZIP directory. Payload CRC/SHA is NOT re-verified here.
#         with zipfile.ZipFile(target) as archive:
#             members = archive.infolist()
#         result = {m.filename: m.file_size for m in members}
#         if len(result) != len(members) or any(n not in files or size != files[n][0] for n, size in result.items()):
#             raise ValueError('Archive entries differ from this snapshot (names/sizes/duplicates).')
#         return result

#     attempts = set()
#     extra_roots = {Path(p).resolve() for p in search_dirs}
#     roots = {export_dir, bundle.parent, *extra_roots}
#     with tqdm(desc='Find export attempts', unit='attempt') as bar:
#         for root in sorted(roots):
#             for directory, children, names in os.walk(root):
#                 children[:] = [n for n in children if not n.startswith('review_bundle_')
#                                and n not in {'images', 'proposals', 'runtime', 'inputs'}]
#                 for name in names:
#                     if ((name.startswith(bundle.name) and ('.zip' in name or name.endswith('.export.json')))
#                             or (root in extra_roots and name == 'results.zip')):
#                         path = Path(directory) / name
#                         if path not in attempts: attempts.add(path); bar.update(1)
#     rows = []
#     covered = set()
#     # Checkpoint tails can be inspected without touching an interrupted work ZIP.
#     for path in tqdm(sorted(p for p in attempts if p.name.endswith('.export.json')), desc='Review checkpoints', unit='attempt'):
#         row = {'path':str(path), 'kind':'checkpoint', 'bytes':0, 'files':0, 'usable':False}
#         try:
#             state = json.loads(path.read_text())
#             work = Path(state['archive']).resolve()
#             covered.add(work)
#             if (state['bundle'] != str(bundle) or state['fingerprint'] != fingerprint
#                     or work.parent != export_dir or not work.name.startswith(bundle.name + '_resume_')
#                     or not work.name.endswith('.work.zip')):
#                 raise ValueError('Checkpoint is for a different snapshot or unsupported work path.')
#             offset = state['directory_offset']
#             if not isinstance(offset,int) or offset < 0 or work.stat().st_size < offset:
#                 raise ValueError('Committed ZIP prefix is missing.')
#             members = checked_members(io.BytesIO(base64.b64decode(state['tail'],validate=True)))
#             row.update(usable=True, bytes=sum(members.values()), files=len(members), state=state)
#         except (OSError, ValueError, KeyError, TypeError, zipfile.BadZipFile) as exc:
#             row['reason'] = str(exc)
#         rows.append(row)
#     for path in tqdm(sorted(p for p in attempts if not p.name.endswith('.export.json') and p.resolve() not in covered), desc='Review ZIP attempts', unit='attempt'):
#         row = {'path':str(path), 'kind':'archive', 'bytes':0, 'files':0, 'usable':False}
#         try:
#             members = checked_members(path)
#             row.update(usable=True, bytes=sum(members.values()), files=len(members))
#         except (OSError, ValueError, zipfile.BadZipFile) as exc:
#             row['reason'] = str(exc) + ' (preserved; no usable checkpoint)'
#         rows.append(row)
#     total = sum(size for size, _ in files.values())
#     rows.sort(key=lambda r:(r['usable'],r['bytes'],r['files'],
#                            r['kind']=='archive' and r['path'].endswith('.zip') and not r['path'].endswith('.work.zip')), reverse=True)
#     print('Export attempts (ZIP directory checks only; not full payload verification):')
#     for row in rows:
#         pct = 100*row['bytes']/total if total else 0
#         print(f"{pct:6.1f}% | {row['files']}/{len(files)} files | {row['kind']} | "
#               f"{'recoverable' if row['usable'] else row.get('reason','unusable')} | {row['path']}")
#     best = next((row for row in rows if row['usable']),None)
#     if best:
#         print('Most export progress:',best['path'])
#         candidate = Path(best['path'])
#         if best['kind']=='archive' and best['files']==len(files) and candidate.suffix=='.zip' and not candidate.name.endswith('.work.zip'):
#             print('Existing complete ZIP (file list/sizes checked):',candidate)
#             return candidate

#     state_path = (Path(best['path']) if best and best['kind']=='checkpoint' else
#                   export_dir / (bundle.name + '_' + fingerprint[:16] + '_' + uuid.uuid4().hex[:8] + '.export.json'))

#     def save_checkpoint(work, directory_offset):
#         # Back up the committed central directory BEFORE the next append overwrites it.
#         # Restoring this tail discards only uncommitted bytes in this owned work ZIP.
#         with work.open('rb') as handle:
#             handle.seek(directory_offset)
#             tail = base64.b64encode(handle.read()).decode('ascii')
#         state = {'bundle': str(bundle), 'fingerprint': fingerprint, 'archive': str(work),
#                  'directory_offset': directory_offset, 'tail': tail}
#         temporary = state_path.with_name(state_path.name + '.' + uuid.uuid4().hex + '.tmp')
#         temporary.write_text(json.dumps(state), encoding='utf-8')
#         temporary.replace(state_path)

#     if best and best['kind']=='checkpoint':
#         state = best['state']
#         work = Path(state['archive']).resolve()
#         if (state['bundle'] != str(bundle) or state['fingerprint'] != fingerprint
#                 or work.parent != export_dir or not work.name.startswith(bundle.name + '_resume_')
#                 or not work.name.endswith('.work.zip')):
#             raise ValueError('Export checkpoint does not match this bundle; existing files were preserved.')
#         offset = state['directory_offset']
#         tail = base64.b64decode(state['tail'], validate=True)
#         if not isinstance(offset, int) or offset < 0 or work.stat().st_size < offset:
#             raise ValueError('Committed ZIP bytes are missing. Existing files were preserved.')
#         with work.open('r+b') as handle:
#             handle.seek(offset)
#             handle.write(tail)
#             handle.truncate(offset + len(tail))
#         print('Resuming saved export:', work)
#     else:
#         work = export_dir / (bundle.name + '_resume_' + uuid.uuid4().hex + '.work.zip')
#         if best:
#             # Clone a readable legacy partial ZIP. Never append to the original attempt.
#             source_path = Path(best['path'])
#             with source_path.open('rb') as source, work.open('xb') as destination, tqdm(
#                     total=source_path.stat().st_size,desc='Preserve previous ZIP progress',unit='B',unit_scale=True,unit_divisor=1024) as bar:
#                 for chunk in iter(lambda:source.read(4*1024*1024),b''):
#                     destination.write(chunk); bar.update(len(chunk))
#             with zipfile.ZipFile(work) as archive: pass
#         else:
#             with zipfile.ZipFile(work, 'x', compression=zipfile.ZIP_STORED) as archive: pass
#         save_checkpoint(work, archive.start_dir)
#         print('New resumable export:', work)

#     with zipfile.ZipFile(work) as archive:
#         committed = {m.filename: m.file_size for m in archive.infolist()}
#     if any(name not in files or size != files[name][0] for name, size in committed.items()):
#         raise ValueError('Checkpoint contains unexpected files. Existing source data were preserved.')
#     pending = sorted(name for name in files if name not in committed)
#     print('Files already exported:', len(committed), '| files remaining:', len(pending))

#     # Store already-compressed PNG/JPEG/NPZ directly. No extra SHA pass or full ZIP reread.
#     with tqdm(total=total, initial=sum(committed.values()), desc='Export saved files',
#               unit='B', unit_scale=True, unit_divisor=1024) as bar:
#         for start in range(0, len(pending), batch_size):
#             batch = pending[start:start + batch_size]
#             with zipfile.ZipFile(work, 'a', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
#                 for name in batch:
#                     path = bundle / name
#                     before = path.stat()
#                     if [before.st_size, before.st_mtime_ns] != files[name]:
#                         raise RuntimeError('Source changed. Stop inference/review writes before exporting.')
#                     written = 0
#                     with path.open('rb') as source, archive.open(name, 'w', force_zip64=True) as destination:
#                         for chunk in iter(lambda: source.read(4*1024*1024), b''):
#                             destination.write(chunk)
#                             written += len(chunk)
#                             bar.update(len(chunk))
#                     after = path.stat()
#                     if written != files[name][0] or [after.st_size, after.st_mtime_ns] != files[name]:
#                         raise RuntimeError('Source changed during export. Stop concurrent writes and rerun.')
#             save_checkpoint(work, archive.start_dir)
#             bar.set_postfix(files_committed=len(committed) + min(start + batch_size, len(pending)))

#     if len(checked_members(work)) != len(files):
#         raise RuntimeError('Final ZIP file list/size check failed; work ZIP and checkpoints preserved.')
#     final = work.with_name(work.name.removesuffix('.work.zip') + '.zip')
#     if final.exists():
#         raise FileExistsError(final)
#     work.rename(final)
#     print('Drive ZIP ready:', final)
#     print('File list/sizes checked; full payload CRC/SHA re-verification skipped for speed.')
#     return final


# if __name__ == '__main__':
#     # After a runtime reset, paste the EXACT saved bundle path here; no earlier cells needed.
#     BUNDLE_TO_EXPORT = globals().get('OUT') or globals().get('RESUME_BUNDLE', '')
#     EXPORT_FOLDER = Path(globals().get('ROOT', '/content/drive/MyDrive/Robotekin2026')) / 'exports'
#     if not BUNDLE_TO_EXPORT or str(BUNDLE_TO_EXPORT) == 'auto':
#         raise ValueError('Set BUNDLE_TO_EXPORT above to your exact saved review_bundle folder.')
#     if not Path('/content/drive/MyDrive').exists():
#         from google.colab import drive
#         drive.mount('/content/drive')
#     export_saved_bundle(BUNDLE_TO_EXPORT, EXPORT_FOLDER,
#         search_dirs=[globals().get('EXPORT_LOCAL_DIR','/content/roboqc_review_exports')],
#         listing_workers=globals().get('EXPORT_LISTING_WORKERS',4),
#         refresh_inventory=globals().get('REFRESH_EXPORT_INVENTORY',False))

In [ ]:
# # New snapshot on every execution; no model weights, tokens or input ZIP duplication.
# (OUT / 'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
# (OUT / 'README.txt').write_text('roboqc-review-1.0: images.json + proposals.json + reviews.json. All mask coordinates refer to images/<image_id>/rgb.png (EXIF-normalized). Originals preserved. Review records are NOT ground truth until explicitly accepted. Multiple instance masks per image, not one merged mask. Raw NPZ includes filtered proposals. Scores are not comparable accuracy. No robot execution. Use matching runtime source and settings for provenance. Model weights and credentials excluded.',encoding='utf-8')
# runtime_out = OUT / 'runtime'
# runtime_out.mkdir(exist_ok=True)
# for relative,source in SOURCES.items():
#     target = runtime_out / relative; target.parent.mkdir(parents=True,exist_ok=True)
#     if not target.exists(): target.write_text(source,encoding='utf-8')
# snapshot = Path(ROOT) / 'exports' / (OUT.name + '_' + uuid.uuid4().hex + '.zip')
# exported = archive_bundle(OUT,snapshot)
# print('Drive results ZIP:',exported)
# print('Input images:',len(samples),'| inference-selected images:',len(active))
# print('Errors:',sum(r['status'] != 'ok' for r in records),'| unreviewed: all until app approval')
